# Introduction
This notebook demonstrates auto-segmentation of the Stem-Cell Rich Region (SCRR) on H&N CT scans for qualitative evaluation using an AI model [1] to locate the masseters, mandible, and parotids, before delineating the SCRR based on a published method [2]. Future work will include validating a published predictive model for xerostomia [3] based on the SCRR.

* Note: Please contact the maintainers of [SMIT](github.com/cerr/smit) to request a link to the pre-trained weights (`weightsUrl`).

### References
1. Jiang, J., Tyagi, N., Tringale, K., Crane, C., & Veeraraghavan, H. Self-supervised 3D anatomy segmentation using self-distilled masked image transformer (SMIT). In International conference on medical image computing and computer-assisted intervention Sept. 2022; pp. 556-566. Cham: Springer Nature Switzerland.
2. van Rijn-Dekker, M.I.; van der Schaaf, A.; Nienhuis, S.W.; Arents-Huls, A.S.; Ger, R.B.; Hamming-Vrieze, O.; Hoebers, F.J.P.; de Ridder, M.; Vigorito, S.; Zwijnenburg, E.M.; et al. Clinical Introduction of Stem Cell Sparing Radiotherapy to Reduce the Risk of Xerostomia in Patients with Head and Neck Cancer. Cancers 2024, 16, 4283. https://doi.org/10.3390/cancers16244283
3. van Rijn-Dekker MI, van Luijk P, Schuit E, van der Schaaf A, Langendijk JA, Steenbakkers RJHM. Prediction of Radiation-Induced Parotid Gland-Related Xerostomia in Patients With Head and Neck Cancer: Regeneration-Weighted Dose. Int J Radiat Oncol Biol Phys. 2023 Nov 1;117(3):750-762. doi: 10.1016/j.ijrobp.2023.04.034. Epub 2023 May 6. PMID: 37150262.

## Download H&N CT dataset
[The HaN-Seg: Head and Neck Organ-at-Risk CT & MR Segmentation Dataset](https://zenodo.org/records/7442914)


In [ ]:
import os
import glob
import shutil
import yaml
import pandas as pd

### Data I/O

In [ ]:
dataDir = r'/path/to/hn_ct/dataset'
scanDir = r'/path/to/hn_ct/ct_only'               # Temp dir: copy of the dataset holding the CT scans only
analysisDir = r'/path/to/logs_and_results'        # Directory for logs and results
installDir = r'/path/to/ai_model_installs'        # Location of the AI H&N CT segmentation model
settingsFile = r'sample_settings/seg_scrr.yaml'   # sample settings
modelNum = 7                                      # CT_HN SMIT model
weightsUrl = 'URL-received-from-the-model-maintainers'

### Example:

## Copy scans to `scanDir`
To speed up the analysis, copy only the CT scans (_\*\_IMG\_CT.nrrd_) to a separate directory, one subfolder per case.

In [ ]:
rows = []
for caseDir in sorted(glob.glob(os.path.join(dataDir, '*'))):
    ctFiles = glob.glob(os.path.join(caseDir, '*_IMG_CT.nrrd'))
    if not os.path.isdir(caseDir) or not ctFiles:
        continue
    caseId = os.path.basename(caseDir)
    os.makedirs(os.path.join(scanDir, caseId), exist_ok=True)
    shutil.copy2(ctFiles[0], os.path.join(scanDir, caseId))
    rows.append({'ID': caseId, 'Scan file': os.path.basename(ctFiles[0])})

### Update the settings file to point to your data and model installation locations

In [ ]:
from validate.core import analysis

with open(settingsFile) as f:
    settings = yaml.safe_load(f)
settings['input']['data_dir'] = scanDir
settings['segment']['install_dir'] = installDir
analysis.saveSettings(settings, analysisDir)

### Run auto-segmentation

In [ ]:
from validate.seg.autoseg_pipeline import installMissingModels

# Install the selected model
installMissingModels([modelNum], installDir, progressCallback=print, force=True, weightsUrl=weightsUrl)

In [ ]:
from validate.run import applyAutoSegToCohort, deriveStructuresForCohort
from validate.logs import ErrorLog, generateSummary

# Stage 1: segment requisite structures for SCRR identification (parotids, masseters, mandible)
errorLog = ErrorLog(analysisDir)
try:
    # Run the SMIT model to segment the masseters, partoids, and mandible.
    stage1Result = applyAutoSegToCohort(analysisDir, errorLog, stageCallback=print,
                                        progressCallback=lambda done, total, pid: print(f'{done}/{total}  {pid}'))
    # Segment the SCRR
    stage2Result = deriveStructuresForCohort(analysisDir, errorLog)
finally:
    errorLog.close()

### Generate a report showing the list of cases processed and structures generated

In [ ]:
summaryFile = generateSummary(analysisDir)

# Derived structures (SCRR) per case
summary = pd.read_csv(summaryFile)
summary[summary['ai_model'] == 'derived']

## Review sample segmentation results

In [ ]:
from cerr import plan_container as pc

sampleCase = r'case_01'
scanFile = os.path.join(scanDir, sampleCase, f'{sampleCase}_IMG_CT.nrrd')
structFile = os.path.join(analysisDir, sampleCase, f'{sampleCase}_SCRR_autoseg.nii.gz')

planC = pc.loadNrrdScan(scanFile)
scanNum = len(planC.scan)-1
planC = pc.loadNiiStructure(structFile, assocScanNum=scanNum, planC=planC, labels_dict={'SCRR':1})
structNumV = [len(planC.structure)-1]

In [ ]:
from cerr.viewer.pycerr_nbviewer import showNB
showNB(planC, scanNum=scanNum, struct_nums=structNumV,
       windowCenter=50, windowWidth=350)

## Run the xerostomia NTCP model
Derive the structures the Xerostomia (grade 2+) model needs (SCRR, Parotids-SCRR, Submandibular glands, Oral cavity) and compute NTCP for each case.

* Note: HaN-Seg has no RT dose, so this demo uses a synthetic dose and the NTCP values are not meaningful. Use each patient's planned dose for a real analysis.

In [ ]:
import numpy as np
import SimpleITK as sitk
from validate import logs
from validate.core import analysis
from validate.run import applyModelToCohort

# HaN-Seg has no RT dose, so this demo writes a SYNTHETIC dose for each case: 70 Gy at the
# centre of the scan, falling off smoothly (Gaussian, 6 cm) to 2 Gy, on a 2.5 mm grid.
# The NTCPs below are therefore NOT meaningful. For a real analysis, list each patient's
# planned dose (NRRD/NIfTI) in the 'Dose file' column of plan_info.csv instead.
planRows = []
for caseId in sorted(os.listdir(scanDir)):
    ctFiles = glob.glob(os.path.join(scanDir, caseId, '*_IMG_CT.nrrd'))
    if not ctFiles:
        continue
    ct = sitk.ReadImage(ctFiles[0])
    spacing = [2.5, 2.5, 2.5]
    size = [int(round(n * s / 2.5)) for n, s in zip(ct.GetSize(), ct.GetSpacing())]
    centre = ct.TransformContinuousIndexToPhysicalPoint([(n - 1) / 2 for n in ct.GetSize()])
    dose = sitk.GaussianSource(sitk.sitkFloat32, size, sigma=[60.0] * 3, mean=centre, scale=68.0,
                               origin=ct.GetOrigin(), spacing=spacing, direction=ct.GetDirection()) + 2.0
    doseFile = f'{caseId}_dose_synthetic.nrrd'
    sitk.WriteImage(dose, os.path.join(scanDir, caseId, doseFile))
    planRows.append({'ID': caseId, 'Scan file': os.path.basename(ctFiles[0]), 'Dose file': doseFile})

# Plan info (scan and dose file per case) and the model's clinical predictor:
# pretreatment_toxicity = pretreatment xerostomia (0 = No, 1 = Yes). Assumed No for all
# cases here; replace with patient data.
planCsv = os.path.join(analysisDir, 'plan_info.csv')
predictorsCsv = os.path.join(analysisDir, 'predictors.csv')
pd.DataFrame(planRows).to_csv(planCsv, index=False)
pd.DataFrame({'ID': [row['ID'] for row in planRows], 'pretreatment_toxicity': 0}).to_csv(predictorsCsv, index=False)

# Settings: the plan info and predictors CSVs, and all four structures the model needs derived
settings = analysis.readSettings(analysisDir)
settings['input']['plan_structures_csv'] = planCsv
settings['input']['predictors_csv'] = predictorsCsv
settings['structures'] = {name: {'source': 'derive'}
                          for name in ['SCRR', 'Parotids-SCRR', 'Submandibular glands', 'Oral cavity']}
analysis.saveSettings(settings, analysisDir)
logs.clearResults(analysisDir, 'model')     # the Model settings changed since the SCRR was derived above

# Derive the structures and compute NTCP for each case
errorLog = ErrorLog(analysisDir)
try:
    ntcpResults = applyModelToCohort(analysisDir, errorLog,
                                     progressCallback=lambda done, total, pid: print(f'{done}/{total} patients done'))
finally:
    errorLog.close()
ntcpResults